In [ ]:
# ====================== 导入库 & 全局配置 ======================
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ========== 核心修复：数据根目录 = 脚本所在文件夹 ==========
# 所有原始CSV文件直接和本脚本放在同一个文件夹里即可
DATA_ROOT = Path(__file__).parent
print(f"当前数据根目录：{DATA_ROOT.resolve()}")
print("请确保所有原始CSV文件都在上述目录中\n")

# ====================== 工具函数 ======================
def read_csv_safe(file_path: Path, table_name: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 读取 {table_name}：{len(df)} 行 × {len(df.columns)} 列")
        return df
    except FileNotFoundError:
        raise FileNotFoundError(
            f"未找到文件：{file_path.resolve()}\n"
            f"请将对应CSV文件放入 {DATA_ROOT.resolve()} 目录后重试"
        )

def validate_primary_key(df: pd.DataFrame, pk_col: str, table_name: str) -> None:
    if df[pk_col].nunique() != len(df):
        dup_num = len(df) - df[pk_col].nunique()
        print(f"⚠️  {table_name} 主键 [{pk_col}] 不唯一，存在 {dup_num} 条重复记录")
    else:
        print(f"✅ {table_name} 主键校验通过")

def check_duplicates(df: pd.DataFrame, table_name: str, subset: list = None) -> pd.DataFrame:
    n_dup = df.duplicated(subset=subset).sum() if subset else df.duplicated().sum()
    print(f"\n==== {table_name} 重复值检查 ====")
    print(f"重复记录数: {n_dup}")
    if n_dup > 0 and subset:
        df = df.drop_duplicates(subset=subset, keep='first').reset_index(drop=True)
        print(f"已按 {subset} 去重，剩余 {len(df)} 行")
    return df

def check_missing(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    miss_count = df.isnull().sum()
    miss_ratio = (miss_count / len(df)).round(4)
    miss_df = pd.DataFrame({"缺失数量": miss_count, "缺失率": miss_ratio})
    miss_df = miss_df[miss_df["缺失数量"] > 0].sort_values("缺失率", ascending=False)
    print(f"\n==== {table_name} 缺失值统计 ====")
    if len(miss_df) > 0:
        miss_df["缺失率"] = miss_df["缺失率"].apply(lambda x: f"{x*100:.2f}%")
        print(miss_df)
    else:
        print("无缺失值")
    return miss_df

# ====================== 1. 读取全部原始表 ======================
print("="*50)
print("开始读取原始数据集")
print("="*50)

customers = read_csv_safe(DATA_ROOT / "olist_customers_dataset.csv", "客户表")
order_items = read_csv_safe(DATA_ROOT / "olist_order_items_dataset.csv", "订单商品明细表")
order_payments = read_csv_safe(DATA_ROOT / "olist_order_payments_dataset.csv", "支付表")
order_reviews = read_csv_safe(DATA_ROOT / "olist_order_reviews_dataset.csv", "评价表")
orders = read_csv_safe(DATA_ROOT / "olist_orders_dataset.csv", "订单主表")
products = read_csv_safe(DATA_ROOT / "olist_products_dataset.csv", "商品表")
sellers = read_csv_safe(DATA_ROOT / "olist_sellers_dataset.csv", "卖家表")
category_trans = read_csv_safe(DATA_ROOT / "product_category_name_translation.csv", "品类翻译表")
geolocation = read_csv_safe(DATA_ROOT / "olist_geolocation_dataset.csv", "地理表")

# 主键校验
print("\n" + "="*50)
print("主键唯一性校验")
print("="*50)
validate_primary_key(orders, "order_id", "订单主表")
validate_primary_key(customers, "customer_id", "客户表")
validate_primary_key(products, "product_id", "商品表")
validate_primary_key(sellers, "seller_id", "卖家表")

# ====================== 2. 数据质量检查 ======================
orders = check_duplicates(orders, "订单主表", subset=["order_id"])
order_items = check_duplicates(order_items, "订单商品明细表", subset=["order_id", "order_item_id"])
order_reviews = check_duplicates(order_reviews, "评价表", subset=["review_id"])
products = check_duplicates(products, "商品表", subset=["product_id"])
customers = check_duplicates(customers, "客户表", subset=["customer_id"])
sellers = check_duplicates(sellers, "卖家表", subset=["seller_id"])
category_trans = check_duplicates(category_trans, "品类翻译表", subset=["product_category_name"])

check_missing(orders, "订单主表")
check_missing(products, "商品表")
check_missing(order_reviews, "评价表")

# ====================== 3. 时间字段格式统一 ======================
time_cols_orders = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in time_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"], errors="coerce")
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"], errors="coerce")

print("\n✅ 时间字段格式转换完成")

# ====================== 4. 一对多表聚合（订单粒度） ======================
print("\n" + "="*50)
print("开始一对多表聚合处理")
print("="*50)

# 4.1 商品明细聚合
order_items_agg = order_items.groupby("order_id").agg(
    price_total=("price", "sum"),
    freight_total=("freight_value", "sum"),
    item_count=("order_item_id", "count"),
    product_id=("product_id", "first"),
    seller_id=("seller_id", "first"),
    shipping_limit_date=("shipping_limit_date", "first")
).reset_index()
order_items_agg["order_value"] = order_items_agg["price_total"] + order_items_agg["freight_total"]

# 4.2 评价聚合
order_reviews_agg = order_reviews.groupby("order_id").agg(
    review_score_avg=("review_score", "mean"),
    review_count=("review_id", "count"),
    review_creation_date=("review_creation_date", "max"),
    review_answer_timestamp=("review_answer_timestamp", "max")
).reset_index()

order_reviews["has_comment_title"] = order_reviews["review_comment_title"].notna().astype(int)
order_reviews["has_comment_message"] = order_reviews["review_comment_message"].notna().astype(int)
review_flag = order_reviews.groupby("order_id").agg({
    "has_comment_title": "max",
    "has_comment_message": "max"
}).reset_index()
order_reviews_agg = order_reviews_agg.merge(review_flag, on="order_id", how="left")

# 4.3 支付聚合
order_payments_agg = order_payments.groupby("order_id").agg(
    payment_total=("payment_value", "sum"),
    main_payment_type=("payment_type", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    payment_type_count=("payment_type", "nunique"),
    total_installments=("payment_installments", "sum")
).reset_index()
order_payments_agg["is_mixed_payment"] = (order_payments_agg["payment_type_count"] > 1).astype(int)

print("✅ 一对多表聚合完成")

# ====================== 5. 构建订单粒度宽表 ======================
df_main = orders.copy()
df_main = df_main.merge(customers, on="customer_id", how="left")
df_main = df_main.merge(order_items_agg, on="order_id", how="left")
df_main = df_main.merge(products, on="product_id", how="left")
df_main = df_main.merge(category_trans, on="product_category_name", how="left")
df_main = df_main.merge(sellers, on="seller_id", how="left")
df_main = df_main.merge(order_reviews_agg, on="order_id", how="left")
df_main = df_main.merge(order_payments_agg, on="order_id", how="left")

print(f"\n多表合并完成，订单粒度宽表维度：{df_main.shape[0]} 行 × {df_main.shape[1]} 列")

# ====================== 6. 基础特征衍生 ======================
# 物流时效
df_main["delivery_hours"] = (
    df_main["order_delivered_customer_date"] - df_main["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df_main["estimated_delivery_hours"] = (
    df_main["order_estimated_delivery_date"] - df_main["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df_main["delivery_deviation_hours"] = df_main["delivery_hours"] - df_main["estimated_delivery_hours"]
df_main["is_delay"] = (df_main["delivery_deviation_hours"] > 0).astype(int)

# 价值占比
df_main["freight_ratio"] = df_main["freight_total"] / df_main["order_value"].replace(0, np.nan)

# 评价标签：未评价保留空值，不强行归类
df_main["is_bad_review"] = np.where(
    df_main["review_score_avg"].isna(), np.nan,
    np.where(df_main["review_score_avg"] <= 3, 1, 0)
)

# 时间维度
df_main["order_year"] = df_main["order_purchase_timestamp"].dt.year
df_main["order_quarter"] = df_main["order_purchase_timestamp"].dt.quarter
df_main["order_month"] = df_main["order_purchase_timestamp"].dt.month
df_main["order_day"] = df_main["order_purchase_timestamp"].dt.day
df_main["order_weekday"] = df_main["order_purchase_timestamp"].dt.weekday
df_main["order_hour"] = df_main["order_purchase_timestamp"].dt.hour
df_main["order_weekofyear"] = df_main["order_purchase_timestamp"].dt.isocalendar().week.astype(int)

print("✅ 业务特征衍生完成")

# ====================== 7. 分层清洗 ======================
print("\n" + "="*50)
print("开始分层数据清洗")
print("="*50)
print(f"清洗前总样本量：{df_main.shape[0]}")

# 仅保留已交付订单
df_main_clean = df_main[df_main["order_delivered_customer_date"].notna()].copy()
print(f"过滤未交付订单后：{len(df_main_clean)} 行")

# 商品属性缺失值：同品类均值填充 → 全局兜底
fill_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in fill_cols:
    df_main_clean[col] = df_main_clean.groupby("product_category_name")[col].transform(
        lambda x: x.fillna(x.mean())
    )
    df_main_clean[col] = df_main_clean[col].fillna(df_main_clean[col].mean())

# 剔除空品类
df_main_clean = df_main_clean[df_main_clean["product_category_name"].notna()].reset_index(drop=True)
print(f"剔除空品类后：{len(df_main_clean)} 行")

# 类别型字段优化内存
cat_cols = [
    "order_status", "customer_state", "product_category_name",
    "product_category_name_english", "seller_state", "main_payment_type"
]
for col in cat_cols:
    if col in df_main_clean.columns:
        df_main_clean[col] = df_main_clean[col].astype("category")

check_missing(df_main_clean, "清洗后订单宽表")

# ====================== 8. 构建商品粒度宽表 ======================
df_item = order_items.merge(orders, on="order_id", how="left")
df_item = df_item.merge(customers, on="customer_id", how="left")
df_item = df_item.merge(products, on="product_id", how="left")
df_item = df_item.merge(category_trans, on="product_category_name", how="left")
df_item = df_item.merge(sellers, on="seller_id", how="left")
df_item = df_item.merge(order_reviews_agg, on="order_id", how="left")

df_item["item_total_value"] = df_item["price"] + df_item["freight_value"]
df_item["order_year"] = df_item["order_purchase_timestamp"].dt.year
df_item["order_month"] = df_item["order_purchase_timestamp"].dt.month

df_item_clean = df_item[df_item["order_delivered_customer_date"].notna()].copy()
df_item_clean = df_item_clean[df_item_clean["product_category_name"].notna()].reset_index(drop=True)

# ====================== 9. 地理表清洗聚合 ======================
geolocation_clean = geolocation.groupby("geolocation_zip_code_prefix").agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first")
).reset_index()

# ====================== 10. RFM 基础表 ======================
rfm = df_main_clean.groupby("customer_id").agg(
    last_purchase_time=("order_purchase_timestamp", "max"),
    frequency=("order_id", "count"),
    monetary=("order_value", "sum")
).reset_index()

max_date = rfm["last_purchase_time"].max()
rfm["recency_days"] = (max_date - rfm["last_purchase_time"]).dt.days

# ====================== 11. 统一导出 ======================
print("\n" + "="*50)
print("开始导出清洗结果")
print("="*50)

df_main_clean.to_csv(DATA_ROOT / "df_main_wide.csv", index=False, encoding="utf-8-sig")
print("✅ 订单粒度宽表已导出：df_main_wide.csv")

df_item_clean.to_csv(DATA_ROOT / "df_item_wide.csv", index=False, encoding="utf-8-sig")
print("✅ 商品粒度宽表已导出：df_item_wide.csv")

geolocation_clean.to_csv(DATA_ROOT / "geolocation_clean.csv", index=False, encoding="utf-8-sig")
print("✅ 地理清洗表已导出：geolocation_clean.csv")

rfm.to_csv(DATA_ROOT / "rfm_base.csv", index=False, encoding="utf-8-sig")
print("✅ RFM基础表已导出：rfm_base.csv")

print("\n" + "="*50)
print("🎉 全部数据清洗与特征工程完成！")
print(f"所有结果文件已保存至：{DATA_ROOT.resolve()}")
print("="*50)